In [ ]:
import uproot
import awkward as ak

import pandas as pd

import matplotlib.pylab as plt
import numpy as np

import hist
from hist import Hist

import time

import myPIDselector

import math

from numba_stats import expon, norm, truncexpon, truncnorm

from iminuit import cost
from iminuit import Minuit, describe
from iminuit.cost import UnbinnedNLL
from iminuit.cost import BinnedNLL
from iminuit.cost import LeastSquares

from scipy.stats import argus
from scipy.stats import fit as _scipy_stats_fit
from scipy.stats import chi2 as chi2_dist
from scipy.optimize import minimize_scalar

import ipywidgets

import os

import scipy.integrate as integrate

%config InlineBackend.figure_formats = ['svg']

MCD = ak.from_parquet("Background_and_signal_SP_modes_All_runs.parquet")
blindD = ak.from_parquet("Data_All_runs_BLINDED.parquet")

B_D1_IDX_FIELD            = 'Bd1Idx'
B_D2_IDX_FIELD            = 'Bd2Idx'

# Lambda daughter fields
LAMBDA_PROTON_IDX_FIELD   = 'Lambda0d1Idx'
LAMBDA_PROTON_LUND_FIELD  = 'Lambda0d1Lund'

 
PROTON_TRKIDX_FIELD       = 'pTrkIdx'
PROTON_SELECTORS_FIELD    = 'pSelectorsMap'
TRK_LUND_FIELD            = 'TRKLund'

MES_FIELD                 = 'BpostFitMes'
DELTAE_FIELD              = 'BpostFitDeltaE'

SPMODE_FIELD              = 'spmode'

plt.close()

def calculate_bits_for_PID_selector(trkidx, trk_selector_map, verbose=0):
    
    bits = None

    # If there is no trk index passed in, just calculate the bits for
    # all of the tracks
    if trkidx is None:
        trkidx = ak.local_index(trk_selector_map)

    # Grab the tracks that map on to the particle/collection we are interested in 
    subset_of_trk_selector_map = trk_selector_map[trkidx]
    if verbose:
        print("values in the subset of the trk selector map")
        print(subset_of_trk_selector_map)
        print()
        
    # We need this to convert the numbers in the selector to binary
    binary_repr_vec = np.vectorize(np.binary_repr)

    # Grab the number of entries in each so we can unflatten this later
    counts = ak.num(subset_of_trk_selector_map)
    
    # Now get the binary representation (as a string) for the flattened subset
    binrep = binary_repr_vec(ak.flatten(subset_of_trk_selector_map), width=32)

    if verbose:
        print("binary representation of selector map")
        print(binrep)
        print()

    # Convert the string to integers
    tempbits = np.array(binrep).astype(int)
    bits = ak.unflatten(tempbits,counts)

    if verbose:
        print("flattened integer representation of selectors as binary (int)")
        print(tempbits)
        print()
        print("unflattened integer representation of selectors as binary (int)")
        print(bits)
        print()

    return bits
def mask_PID_selection(bits, selector, pid_map_object):


    bit_to_look_for = pid_map_object.selectors.index(selector)

    place = int(math.pow(10,bit_to_look_for))
    #print(place)

    mask = bits // place % 10

    mask_bool = ak.values_astype(mask,bool)

    return mask_bool
def get_info_for_PID_masks(ak_arr, verbosity=0):

    # Proton and pion information from the Lambda decay
    # These are the index of the proton (d1) and pion (d2) in those lists
    L1d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd1Idx]
    L1d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd1Idx]

    L2d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd2Idx]
    L2d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd2Idx]


    
     
    #d1lund = ak_arr['Lambda0d1Lund']
    #d2lund = ak_arr['Lambda0d2Lund']
    
    #Bd2idx = ak_arr['Bd2Idx']
    #Bd2lund = ak_arr['Bd2Lund']
    
    trkidx_proton = ak_arr['pTrkIdx']
    trk_selector_map_proton = ak_arr['pSelectorsMap']
    
    trkidx_pion = ak_arr['piTrkIdx']
    trk_selector_map_pion = ak_arr['piSelectorsMap']

    indices_and_maps = {}
    indices_and_maps['L1d1idx'] = L1d1idx
    indices_and_maps['L1d2idx'] = L1d2idx
    indices_and_maps['L2d1idx'] = L2d1idx
    indices_and_maps['L2d2idx'] = L2d2idx
    #indices_and_maps['Bd2idx'] = Bd2idx

    indices_and_maps['trkidx_proton'] = trkidx_proton
    indices_and_maps['trk_selector_map_proton'] = trk_selector_map_proton

    indices_and_maps['trkidx_pion'] = trkidx_pion
    indices_and_maps['trk_selector_map_pion'] = trk_selector_map_pion

    return indices_and_maps
def PID_masks(indices_and_maps, \
              lam1p_selector,\
              lam1pi_selector, \
              lam2p_selector,\
              lam2pi_selector, \
             verbosity=0):
                #change Bpselector
    L1d1idx = indices_and_maps['L1d1idx']
    L1d2idx = indices_and_maps['L1d2idx']
    L2d1idx = indices_and_maps['L2d1idx']
    L2d2idx = indices_and_maps['L2d2idx']
    #Bd2idx = indices_and_maps['Bd2idx']

    trkidx_proton = indices_and_maps['trkidx_proton']
    trk_selector_map_proton = indices_and_maps['trk_selector_map_proton']

    trkidx_pion = indices_and_maps['trkidx_pion']
    trk_selector_map_pion = indices_and_maps['trk_selector_map_pion']
    
    # Proton
    pbits = calculate_bits_for_PID_selector(trkidx_proton, trk_selector_map_proton, verbose=verbosity)
    # Pion
    pibits = calculate_bits_for_PID_selector(trkidx_pion, trk_selector_map_pion, verbose=verbosity)
    
    
    mask_bool_protonL1 = mask_PID_selection(pbits[L1d1idx], lam1p_selector, pps)
        
    mask_bool_pionL1 = mask_PID_selection(pibits[L1d2idx], lam1pi_selector, pips)

    mask_bool_protonL2 = mask_PID_selection(pbits[L2d1idx], lam2p_selector, pps)
        
    mask_bool_pionL2 = mask_PID_selection(pibits[L2d2idx], lam2pi_selector, pips)

    return mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2
def indices_to_booleans(indices, array_to_slice):
    """Convert ragged integer indices to a boolean mask over array_to_slice."""
    whole_set, in_set = ak.unzip(ak.cartesian(
        [ak.local_index(array_to_slice), indices], nested=True
    ))
    return ak.any(whole_set == in_set, axis=-1)
def ragged_element_pick(ragged_arr, flat_idx):
    """
    For each event i, return ragged_arr[i][flat_idx[i]].

    Parameters
    ----------
    ragged_arr : ak.Array, shape [n_events, var]
    flat_idx   : array-like, shape [n_events]  (integer index per event)

    Returns
    -------
    np.ndarray, shape [n_events]
    """
    flat_content = ak.flatten(ragged_arr)
    counts       = ak.num(ragged_arr)

    offsets = np.zeros(len(counts) + 1, dtype=np.int64)
    np.cumsum(ak.to_numpy(counts), out=offsets[1:])

    idx_np = ak.to_numpy(flat_idx).astype(np.int64)
    global_indices = offsets[:-1] + idx_np

    return ak.to_numpy(flat_content)[global_indices]
def build_antilambda_antimask(ak_arr, pps,
                               selector,
                               verbose=0):
    """
    Antibaryon veto for B0 -> Lambda Lambda (BNV).

    REQUIRES: single-B-candidate cut already applied (exactly one B per event).

    Returns
    -------
    charge_test : awkward array of booleans
        Per-track flag — True where a rest-of-event track is an antiproton.
    """

    n = len(ak_arr)

     # 1.  Squeeze B-candidate dimension
    #bd1idx = ak.to_numpy(ak_arr[B_D1_IDX_FIELD][:, 0])   # scalar per event
    bd1idx = ak.to_numpy(ak.firsts(ak_arr[B_D1_IDX_FIELD], axis=1))

    #bd2idx = ak.to_numpy(ak_arr[B_D2_IDX_FIELD][:, 0])
    bd2idx = ak.to_numpy(ak.firsts(ak_arr[B_D2_IDX_FIELD], axis=1))

    if verbose:
        print(f"bd1idx  {bd1idx[:5]}")
        print(f"bd2idx  {bd2idx[:5]}")
        print()

     
    # 2.  Chain: B daughter Lambda index -> that Lambda's proton index
     
    all_lam_proton_idx  = ak_arr[LAMBDA_PROTON_IDX_FIELD]
    all_lam_proton_lund = ak_arr[LAMBDA_PROTON_LUND_FIELD]

    # Proton-list index for each B-daughter Lambda (flat numpy, one per event)
    lam1_pidx = ragged_element_pick(all_lam_proton_idx, bd1idx)
    lam2_pidx = ragged_element_pick(all_lam_proton_idx, bd2idx)

    if verbose:
        print(f"lam1_pidx (proton-list index from Λ₁)  {lam1_pidx[:5]}")
        print(f"lam2_pidx (proton-list index from Λ₂)  {lam2_pidx[:5]}")
        print()

     #     qlam1 = +1 for Λ → p π⁻,  -1 for Λ̄ → p̄ π⁺
    lam1_lund = ragged_element_pick(all_lam_proton_lund, bd1idx)
    lam2_lund = ragged_element_pick(all_lam_proton_lund, bd2idx)

    qlam1 = np.sign(lam1_lund).astype(float)   # shapes
    qlam2 = np.sign(lam2_lund).astype(float)

    if verbose:
        print(f"qlam1 (charge of Λ₁ proton)  {qlam1[:5]}")
        print(f"qlam2 (charge of Λ₂ proton)  {qlam2[:5]}")
        print()

    qtrk = ak_arr[TRK_LUND_FIELD] / np.abs(ak_arr[TRK_LUND_FIELD])

     # 4.  Map proton-list indices to track-list indices
    trk_selector_map = ak_arr[PROTON_SELECTORS_FIELD]
    trkidx_proton    = ak_arr[PROTON_TRKIDX_FIELD]

    # Pick the track index for each signal proton (flat numpy)
    lam1_trkidx_flat = ragged_element_pick(trkidx_proton, lam1_pidx)
    lam2_trkidx_flat = ragged_element_pick(trkidx_proton, lam2_pidx)

    # Wrap as length-1 ragged for indices_to_booleans
    lam1_trkidx = ak.unflatten(lam1_trkidx_flat, 1)
    lam2_trkidx = ak.unflatten(lam2_trkidx_flat, 1)

    if verbose:
        print(f"lam1_trkidx  {lam1_trkidx[:5]}")
        print(f"lam2_trkidx  {lam2_trkidx[:5]}")
        print()

    bool_idx1 = indices_to_booleans(lam1_trkidx, trk_selector_map)
    bool_idx2 = indices_to_booleans(lam2_trkidx, trk_selector_map)

    if verbose:
        print(f"bool_idx1 (Λ₁ proton)  {bool_idx1[:3]}")
        print(f"bool_idx2 (Λ₂ proton)  {bool_idx2[:3]}")
        print()

    
    # 5.  PID selection on ALL proton candidates  
    pbits     = calculate_bits_for_PID_selector(None, trk_selector_map)
    mask_bool = mask_PID_selection(pbits, selector, pps)

    if verbose:
        print(f"Tracks passing {selector}:  {ak.sum(ak.num(pbits[mask_bool]))}")
        print()

     # 6.  Charge test on REST-OF-EVENT proton candidates
     
    other_mask  = ~(bool_idx1 | bool_idx2) & mask_bool
    charge_test = qtrk[other_mask] == -qlam1

    if verbose:
        print(f"charge_test  {charge_test[:3]}")

   
    have_opp       = ak.num(charge_test[charge_test])
    nhave_opp      = len(have_opp[have_opp > 0])
    ndont_have_opp = len(have_opp[have_opp == 0])

    return charge_test
def apply_antibaryon_veto(ak_arr, pps, selector):
    """Return a per-event boolean: True = passes veto (no antiproton found)."""
    ct       = build_antilambda_antimask(ak_arr, pps, selector=selector, verbose=0)
    have_opp = ak.num(ct[ct])
    return have_opp == 0
# 1. Define the polynomial model functions
def constant_model(x, c):
    return np.ones_like(x) * c

def linear_model(x, m, c):
    return m * x + c

def quadratic_model(x, a, b, c):
    return a * x**2 + b * x + c

# 2. Function to compute fit parameters, uncertainties, chi-squared, and NDF
def fit_and_analyze(x, y, y998, y1005, degree):
    scaleD = {"998": 424.3/1719*1.84, "1005":424.3/868.3*1.84, "-999":1,'1235':424.3/1289*1.84,'1237':424.3/1306*1.84}
    """
    Fits a polynomial of specified degree to x and y data, 
    weighted by Poisson errors (sigma = sqrt(y)).
    """
    # Poisson errors on counts; avoid division by zero if a bin is empty
    if len(y998)==0:
        y_errors = np.sqrt(y)
        y_errors = np.where(y_errors == 0, 1.0, y_errors)
        weights = 1.0 / y_errors
    else:
        y998_RELerrors = np.where(y998 != 0, np.sqrt(y998) / y998, 0.0)
        y1005_RELerrors = np.where(y1005 != 0, np.sqrt(y1005) / y1005, 0.0)

        y_errors = np.sqrt((y998_RELerrors*(y998*scaleD['998']))**2 + (y1005_RELerrors*(y1005*scaleD['1005']))**2)
        y_errors = np.where(y_errors == 0, 1.0, y_errors)
        weights = 1.0 / y_errors


    
    # Perform weighted polynomial fit
    # cov=True returns the covariance matrix of the parameters
    params, cov = np.polyfit(x, y, deg=degree, w=weights, cov='unscaled')
    
    # Parameter uncertainties are the square roots of the diagonal elements
    param_errors = np.sqrt(np.diag(cov))
    
    # Calculate predicted Y values from the fit
    y_pred = np.polyval(params, x)
    
    # Calculate Chi-Squared: Sum of ((y_observed - y_predicted) / y_error)^2
    chi_sq = np.sum(((y - y_pred) / y_errors) ** 2)
    
    # Degrees of Freedom = Number of data points - Number of fitted parameters
    num_params = degree + 1
    ndf = len(x) - num_params
    
    return params, param_errors, chi_sq, ndf

pps = myPIDselector.PIDselector("p")
pips = myPIDselector.PIDselector("pi")
pidChoiceDict = {}
for plotVal in range(5):
    pidChoice = 1
    if pidChoice == 1:
        myPMask = "SuperLooseKMProtonSelection"
        myPShort = "SL Pro"
        myPiMask = "SuperLooseKMPionMicroSelection"
        myPiShort = 'SL Pi'
        pidChoiceDict[pidChoice] = (myPShort,myPiShort)
    elif pidChoice == 157:
        myPMask = "VeryLooseKMProtonSelection"
        myPShort = "VL Pro"
        myPiMask = "VeryLooseKMPionMicroSelection"
        myPiShort = 'VL Pi'
        pidChoiceDict[pidChoice] = (myPShort,myPiShort)
    elif pidChoice == 202:
        myPMask = "VeryLooseKMProtonSelection"
        myPShort = "VL Pro"
        myPiMask = "TightKMPionMicroSelection"
        myPiShort = 'T   Pi'
        pidChoiceDict[pidChoice] =(myPShort,myPiShort)
    elif pidChoice == 112:
        myPMask = "VeryLooseKMProtonSelection"
        myPShort = "VL Pro"
        myPiMask = "LooseKMPionMicroSelection"
        myPiShort = 'L   Pi'
        pidChoiceDict[pidChoice] = (myPShort,myPiShort)
    elif pidChoice == 256:
        myPMask = "LooseKMProtonSelection"
        myPShort = "L   Pro"
        myPiMask = "VeryLooseKMPionMicroSelection"
        myPiShort = 'VL Pi'
        pidChoiceDict[pidChoice] = (myPShort,myPiShort)
    elif pidChoice == 121:
        myPMask = "SuperLooseKMProtonSelection"
        myPShort = "SL Pro"
        myPiMask = "SuperTightKMPionMicroSelection"
        myPiShort = 'ST Pi'
        pidChoiceDict[pidChoice] = (myPShort,myPiShort)
    elif pidChoice == -1:
        myPMask = "TightKMProtonSelection"
        myPShort = "T   Pro"
        myPiMask = "VeryLooseKMPionMicroSelection"
        myPiShort = 'VL Pi'
        pidChoiceDict[pidChoice] = (myPShort,myPiShort)
    elif pidChoice == -2:
        myPMask = "VeryTightKMProtonSelection"
        myPShort = "VT Pro"
        myPiMask = "VeryLooseKMPionMicroSelection"
        myPiShort = 'VL Pi'
        pidChoiceDict[pidChoice] = (myPShort,myPiShort)
    elif pidChoice == -3:
        myPMask = "SuperTightKMProtonSelection"
        myPShort = "ST Pro"
        myPiMask = "VeryLooseKMPionMicroSelection"
        myPiShort = 'VL Pi'
        pidChoiceDict[pidChoice] = (myPShort,myPiShort)
    elif pidChoice == 0:
        myPMask =  "SuperLooseKMProtonSelection"
        myPiMask = "SuperLooseKMPionMicroSelection"
        pidChoiceDict[pidChoice] = ("None","None")
    data = MCD
    numBCan = data['nB']
    bCanMask = (ak.num(data['BMass'])==1)
    d1b = data[bCanMask]
    indices_and_maps = get_info_for_PID_masks(d1b, verbosity=0)
    mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2 = PID_masks(indices_and_maps, \
                lam1p_selector=myPMask, \
                lam1pi_selector=myPiMask, \
                lam2p_selector=myPMask, \
                lam2pi_selector=myPiMask, \
                verbosity=0)
    maxSigCut,minSigCut = (10,10)
    mes = d1b['BpostFitMes']
    de = d1b['BpostFitDeltaE']
    meslo = 5.2
    mesloS = 5.27
    meshiS = 5.3
    DeltaElo = -0.2
    DeltaEhi = 0.2
    DeltaEloS = -0.07
    DeltaEhiS = 0.07
    mask_pid = mask_bool_pionL1 & mask_bool_protonL1 & mask_bool_pionL2 & mask_bool_protonL2
    preSel_fit = (mes>meslo)&(de<DeltaEhi)&(de>DeltaElo)
    mask_sig = (mes>=mesloS) & (mes<=meshiS) & (de>=DeltaEloS) & (de<=DeltaEhiS)
    mask_sideband = (((mes<mesloS) | (mes>meshiS)) | ((de<DeltaEloS) | (de>DeltaEhiS)))
    lamconflsig = d1b['Lambda0postFitFlightSignificance']
    lamuncmass = d1b['Lambda0_unc_Mass']
    BCTT = d1b['BCosThetaT']
    BCTS = d1b['BCosThetaS']
    lammass_world_average = 1.115683
    width = 0.003
    lo = lammass_world_average - width
    hi = lammass_world_average + width
    mask_fl = (lamconflsig>=minSigCut)
    mask_upper_fl = (lamconflsig>=maxSigCut)
    flightSigMask = (ak.all(mask_fl,axis=1))&(ak.any(mask_upper_fl, axis=1))
    lamuncmassMask = (lamuncmass[:,0]>=lo)&(lamuncmass[:,0]<=hi)& (lamuncmass[:,1]>=lo)&(lamuncmass[:,1]<=hi)
    vetoMask = apply_antibaryon_veto(d1b, pps, 'LooseKMProtonSelection')
    in_signal = preSel_fit&mask_sig
    in_sideband = preSel_fit&mask_sideband

    scaleD = {"998": 424.3/1719*1.84, "1005":424.3/868.3*1.84, "-999":1,'1235':424.3/1289*1.84,'1237':424.3/1306*1.84}
    #scaleD = {"998": 424.3/1719, "1005":424.3/868.3, "-999":1,'1235':424.3/1289,'1237':424.3/1306}

    effDict = {}
    n_srD = {}
    n_sbD = {}
    flatDict = {}
    for sp in ['998', '1005']:
        spmask = (d1b['spmode']==sp)
        bcts_cut = 95
        bctt_cut = 82

        bcttMask = (BCTT<bctt_cut/100) 
        bctsMask = (BCTS<bcts_cut/100)
        deMask = (de >= DeltaEloS) & (de <= DeltaEhiS)
        
        cut_mask = preSel_fit&spmask&deMask&mask_pid&lamuncmassMask&flightSigMask&vetoMask#&bcttMask&bctsMask

        
        mesP = mes[cut_mask]

        flatDict[sp] = ak.to_numpy(ak.flatten(mesP))

    bkgList = [flatDict['998'], flatDict['1005']]
    weightsList = [np.ones_like(flatDict['998']) * scaleD['998'], np.ones_like(flatDict['1005']) * scaleD['1005']]
    #1.814
    labels = ['SP 998', 'SP 1005']
    colors = ['blue', 'royalblue']

    bin_edges = np.linspace(5.2, 5.29, 36)

    xr = (5.2, 5.3)


    plt.figure(figsize = (8,6))
    #n, xe
    counts, bins, patches = plt.hist(
            bkgList, 
            bins=bin_edges, 
            weights = weightsList,
            stacked = True,
            label = labels,
            color=colors, 
            edgecolor='black',
            alpha = 0.6
        )

    y998, _ = np.histogram(flatDict['998'], bins=bin_edges)
    y1005, _ = np.histogram(flatDict['1005'], bins=bin_edges)
    
    n = counts[1]
    xe = bins
    
    bin_centers = (bins[:-1] + bins[1:]) / 2 
    if True:
        cx = bin_centers 

        dx = np.diff(xe)

        
        def line(x, α, β):
            return α + x * β
        
        cdf = np.cumsum(n)

        cdf_normalized = cdf / cdf[-1]

        #least_squares = LeastSquares(cx, n, 0.1, line)

        def linear_cdf(x, slope):
            """ 
            A vectorized mathematical model. 
            'slope' is the parameter iminuit will change.
            """
            a, b = 5.20, 5.30
            u = (x - a) / (b - a)  # Normalized x between 0 and 1
            
            # Mathematical formula for a varying linear background CDF
            return u + slope * (u**2 - u)
        
        def quadratic_cdf(x, alpha, beta):
            """
            Vectorized Quadratic CDF. 
            alpha controls the linear slope tilt.
            beta controls the quadratic curvature (parabola).
            """
            # Map x onto a [0, 1] domain
            A, B = 5.20, 5.30
            u = (x - A) / (B - A)
            
            # Vectorized analytical integral equation
            return u + alpha * (u**2 - u) + beta * (2 * u**3 - 3 * u**2 + u)

        def normExpon_cdf(xe, z, mu, sigma, tau):
            return z * truncnorm.cdf(xe, *xr, mu, sigma) + (1 - z) * truncexpon.cdf(xe, *xr, 0, tau)

            #return (1 - z) * truncexpon.cdf(xe, *xr, 0, tau)

        def argus_cdf_windowed(xval, chi):
            m0 = 5.29
            xmin = 5.2
            xmax = m0
            raw = argus.cdf(xval, chi, loc=0, scale=m0)
            lo  = argus.cdf(xmin, chi, loc=0, scale=m0)
            hi  = argus.cdf(xmax, chi, loc=0, scale=m0)
            return (raw - lo) / (hi - lo)

        def argus_cdf_windowed_extended(xval, s, chi):
            m0 = 5.29
            xmin = 5.2
            xmax = 5.29
            raw = argus.cdf(xval, chi, loc=0, scale=m0)
            lo  = argus.cdf(xmin, chi, loc=0, scale=m0)
            hi  = argus.cdf(xmax, chi, loc=0, scale=m0)
            return s * (raw - lo) / (hi - lo)
        
        def argus_events_in_region(chi, m0, xmin, xmax, a, b, N_total):
            lo = argus.cdf(xmin, chi, loc=0, scale=m0)
            hi = argus.cdf(xmax, chi, loc=0, scale=m0)
            frac = (argus.cdf(b, chi, loc=0, scale=m0) - argus.cdf(a, chi, loc=0, scale=m0)) / (hi - lo)
            return N_total * frac
        
        def extended_argus_events_in_region(chi, a, b, s):
            return ((argus_cdf_windowed_extended(b, s, chi) - argus_cdf_windowed_extended(a, s, chi)))
        
        def argus_events_in_region_with_error(chi_hat, sigma_chi, m0, xmin, xmax, a, b,
                                            N_total, N_total_err=0.0, eps=1e-4):
            
            N = argus_events_in_region(chi_hat, m0, xmin, xmax, a, b, N_total)
        
            n_plus = argus_events_in_region(chi_hat + eps, m0, xmin, xmax, a, b, N_total)
            n_minus = argus_events_in_region(chi_hat - eps, m0, xmin, xmax, a, b, N_total)
            dN_dchi = (n_plus - n_minus) / (2 * eps)
        
            var_from_chi = (dN_dchi * sigma_chi) ** 2
            var_from_N = (N / N_total * N_total_err) ** 2 if N_total > 0 else 0.0
            sigma_N = float(np.sqrt(var_from_chi + var_from_N))
        
            return float(N), sigma_N
        
        def mixture_cdf(xe, z, low, high, mu, sigma, tau):
            """
            f(xe) = z * truncnorm.cdf(xe, a, b, mu, sigma)
                + (1 - z) * truncexpon.cdf(xe, b_exp, 0, tau)
        
            where (a, b) are the *standardized* truncnorm bounds derived from the
            physical window [low, high], and b_exp is the standardized truncexpon
            upper bound. truncexpon's loc is fixed at 0 (per the original snippet),
            so its lower truncation is always 0 regardless of `low`.
        
            Parameters
            ----------
            xe : float
                Point at which the mixture CDF is evaluated (the integration variable).
            z : float
                Mixture weight for the truncated-normal component, 0 <= z <= 1.
            low, high : float
                Physical truncation window shared by both components (high is also
                used as the truncexpon upper bound; low only affects the normal
                component, since truncexpon's loc is pinned to 0).
            mu, sigma : float
                loc / scale for the truncated normal.
            tau : float
                scale for the truncated exponential (loc fixed at 0).
        
            Returns
            -------
            float
            """
            a, b = low, high
        
            return (
                z * truncnorm.cdf(xe, a, b, loc=mu, scale=sigma)
                + (1 - z) * truncexpon.cdf(xe, 0, b, loc=a, scale=tau)
            )
        
        def integrate_mixture_cdf(z, low, high, mu, sigma, tau, N_total, lower, upper):
            """
            Integrate mixture_cdf over [lower, upper] (defaults to 5.27 -> 5.3).
        
            Returns
            -------
            (value, abs_error) : tuple of float
                The integral estimate and scipy's estimated absolute error.
            """
            value, abs_error = integrate.quad(
                mixture_cdf,
                lower,
                upper,
                args=(z, low, high, mu, sigma, tau),
            )
            return value, abs_error
 
        y998_RELerrors = np.where(y998 != 0, np.sqrt(y998) / y998, 0.0)
        y1005_RELerrors = np.where(y1005 != 0, np.sqrt(y1005) / y1005, 0.0)

        vn = (y998_RELerrors**2 + y1005_RELerrors**2)*n
        vn = np.where(vn == 0, 1, vn)


        if plotVal == 0:
            c = cost.BinnedNLL(np.transpose((n,vn)), xe, argus_cdf_windowed)
            m = Minuit(c, chi = 1.2)
            m.strategy = 1
            m.limits['chi'] = (0, 20)
            m.migrad()
            m.hesse()
            model_cdf = argus_cdf_windowed(xe, m.values['chi'])
            expected = n.sum()*np.diff(model_cdf) 
            plt.errorbar(cx, n, np.sqrt(vn), fmt="ok")                            # expected counts per bin
            plt.stairs(expected, xe, fill=False, color="red", linewidth=2.5)
            plt.show()
            #m.visualize()
            print('Chi:')
            print(m.values['chi'])
            m.minos()
            lo = m.merrors["chi"].lower   # negative number — this is an offset, not the bound itself
            hi = m.merrors["chi"].upper   # positive number
            print(f"chi = {m.values['chi']:.4f} [{m.values['chi']+lo:.4f}, {m.values['chi']+hi:.4f}]")
            chi_hat = m.values['chi']
            sigma_chi = 10
            m0, xmin, xmax = 5.29, 5.2, 5.29
            N_total = float(n.sum())
            a, b = 5.2, 5.27
            pred, pred_err = argus_events_in_region_with_error(chi_hat, sigma_chi, m0, xmin, xmax, a, b, N_total)
            print(f"predicted ARGUS events in [{a}, {b}]: {pred:.1f} +/- {pred_err:.1f}")
            print(f"actual count in that region:    {sum(n[:-12])}")
            a, b = 5.27, 5.29
            pred, pred_err = argus_events_in_region_with_error(chi_hat, sigma_chi, m0, xmin, xmax, a, b, N_total)
            print(f"predicted ARGUS events in [{a}, {b}]: {pred:.1f} +/- {pred_err:.1f}")
            print(f"actual count in that region:    {sum(n[-12:])}")

        if plotVal == 1:
            c = cost.ExtendedBinnedNLL(np.transpose((n,vn)), xe, argus_cdf_windowed_extended)
            m = Minuit(c, s=20, chi = 5)
            m.strategy = 1
            m.limits['chi'] = (0, 15)
            m.limits['s'] = (1, 27)

            m.migrad()
            m.hesse()
            model_cdf = argus_cdf_windowed_extended(xe, m.values['s'], m.values['chi'])
            expected = np.diff(model_cdf) 
            plt.errorbar(cx, n, np.sqrt(vn), fmt="ok")                            # expected counts per bin
            plt.stairs(expected, xe, fill=False, color="red", linewidth=2.5)
            plt.show()
            #m.visualize()
            print('Chi:')
            print(m.values['chi'])
            print(m.values['s'])
            m.minos()
            lo = m.merrors["chi"].lower   # negative number — this is an offset, not the bound itself
            hi = m.merrors["chi"].upper   # positive number
            print(f"chi = {m.values['chi']:.4f} [{m.values['chi']+lo:.4f}, {m.values['chi']+hi:.4f}]")
            chi_hat_ext = m.values['chi']
            s = m.values['s']
            a, b = 5.2, 5.27
            predExt = extended_argus_events_in_region(chi_hat_ext, a, b, s)
            print(f"predicted extended ARGUS events in [{a}, {b}]: {predExt:.1f}")
            print(f"actual count in that region:    {sum(n[:-12])}")
            a, b = 5.27, 5.29
            predExt = extended_argus_events_in_region(chi_hat_ext, a, b, s)
            print(f"predicted extended ARGUS events in [{a}, {b}]: {predExt:.1f}")
            print(f"actual count in that region:    {sum(n[-12:])}")

        if plotVal == 2:
            c = cost.BinnedNLL(np.transpose((n,vn)), xe, normExpon_cdf)
            m = Minuit(c, z=0.4, mu = 5.1, sigma = 0.11, tau = 2)
            m.limits["z"] = (0, 1)
            m.limits["mu"] = (0,None)
            m.limits["sigma", "tau"] = (0.01, None)
            m.migrad()
            m.hesse()
            model_cdf = normExpon_cdf(xe, m.values['z'], m.values['mu'],m.values['sigma'], m.values['tau'])
            expected = n.sum()*np.diff(model_cdf) 
            plt.errorbar(cx, n, np.sqrt(vn), fmt="ok")                            # expected counts per bin
            plt.stairs(expected, xe, fill=False, color="red", linewidth=2.5)
            plt.show()
            #m.visualize()
            print('mu, sigma, tau:')
            print(m.values['mu'])
            print(m.values['sigma'])
            print(m.values['tau'])
            z = m.values['z']
            low = 5.2
            high = 5.29
            mu= m.values['mu']
            sigma = m.values['sigma']
            tau = m.values['tau']
            N_total = np.sum(n)
            integration_start = 5.2
            integration_end = 5.27
            val, err = integrate_mixture_cdf(z, low, high, mu, sigma, tau, N_total, integration_start, integration_end)
            print(f"NORM + EXPON MC\nIntegral value in [{a}, {b}]: {val}")
            integration_start = 5.27
            integration_end = 5.29
            val, err = integrate_mixture_cdf(z, low, high, mu, sigma, tau, N_total, integration_start, integration_end)
            print(f"NORM + EXPON MC\nIntegral value in [{a}, {b}]: {val}")


        if plotVal == 3:
            c = cost.BinnedNLL(np.transpose((n,vn)), xe, linear_cdf)
            m = Minuit(c, slope = 0)
            m.limits["slope"] = (None, 0) 
            m.migrad()
            m.hesse()
            model_cdf = linear_cdf(xe, m.values['slope'])
            expected = n.sum()*np.diff(model_cdf) 
            plt.errorbar(cx, n, np.sqrt(vn), fmt="ok")                            # expected counts per bin
            plt.stairs(expected, xe, fill=False, color="red", linewidth=2.5)
            plt.show()
            #m.visualize()
            print('Slope:')
            print(m.values['slope'])
        if plotVal == 4:
            c = cost.BinnedNLL(np.transpose((n,vn)), xe, quadratic_cdf)
            m = Minuit(c, alpha = 0, beta = 0)
            m.limits["alpha"] = (-1, 1.0)
            m.limits["beta"] = (-1, 1)
            m.migrad()
            m.hesse()
            model_cdf = quadratic_cdf(xe, m.values['alpha'], m.values['beta'])
            expected = n.sum()*np.diff(model_cdf) 
            plt.errorbar(cx, n, np.sqrt(vn), fmt="ok")                            # expected counts per bin
            plt.stairs(expected, xe, fill=False, color="red", linewidth=2.5)
            plt.show()
            #m.visualize()
            print('Alpha, beta:')
            print(m.values['alpha'])
            print(m.values['beta'])



        #y_errs = np.sqrt((y998_RELerrors*(y998*scaleD['998']/0.050004))**2 + (y1005_RELerrors*(y1005*scaleD['1005']/0.050004))**2)
        y_errs = np.sqrt((y998_RELerrors*(y998*scaleD['998']))**2 + (y1005_RELerrors*(y1005*scaleD['1005']))**2)
        y_errs = np.where(y_errs == 0, 0, y_errs)


        plt.show()
        m
        print(m)

    
    if False:
                
        def _expected_counts(chi_param, bin_edges, total_counts):
            """Expected counts per bin from ARGUS(chi, loc=0, scale=1), normalized to total_counts."""
            cdf_vals = argus.cdf(bin_edges, chi_param)
            bin_probs = np.diff(cdf_vals)
            return total_counts * bin_probs
        
        
        def _binned_chi2(counts, expected):
            """Pearson chi-square statistic using expected counts as the variance estimate."""
            expected_safe = np.where(expected <= 0, 1e-10, expected)
            return float(np.sum((counts - expected) ** 2 / expected_safe))
        
        
        def fit_argus_chi2(counts, bin_edges, chi_bounds=(0.05, 20.0)):
            """
            Fit chi by directly minimizing the binned chi-square statistic.
        
            Parameters
            ----------
            counts : array_like, shape (n_bins,)
                Observed histogram counts.
            bin_edges : array_like, shape (n_bins + 1,)
                Bin edges; must lie within [0, 1].
            chi_bounds : tuple
                Search bounds for chi during the 1-D minimization.
        
            Returns
            -------
            dict: chi, chi2, dof, p_value, expected (expected counts per bin at best fit)
            """
            counts = np.asarray(counts, dtype=float)
            bin_edges = np.asarray(bin_edges, dtype=float)
            total = counts.sum()
        
            def objective(chi_param):
                expected = _expected_counts(chi_param, bin_edges, total)
                return _binned_chi2(counts, expected)
        
            res = minimize_scalar(objective, bounds=chi_bounds, method='bounded')
            best_chi = float(res.x)
            best_chi2 = float(res.fun)
            dof = len(counts) - 1  # one free parameter (chi)
            p_value = float(chi2_dist.sf(best_chi2, dof))
        
            return {
                'chi': best_chi,
                'chi2': best_chi2,
                'dof': dof,
                'p_value': p_value,
                'expected': _expected_counts(best_chi, bin_edges, total),
            }
        
        
        def fit_argus_mle(counts, bin_edges, chi_bounds=(0.05, 20.0)):
            """
            Fit chi via maximum likelihood on pseudo-samples reconstructed from the
            histogram (bin midpoints repeated by their rounded counts), then compute
            the binned chi2 as a goodness-of-fit check on that fit.
        
            Parameters
            ----------
            counts : array_like, shape (n_bins,)
            bin_edges : array_like, shape (n_bins + 1,)
                Must lie within [0, 1].
            chi_bounds : tuple
                Search bounds for chi during the MLE optimization. Unlike plain
                argus.fit(), this is enforced via scipy.stats.fit(..., bounds=...),
                so the result can't wander off to a degenerate near-zero chi.
        
            Returns
            -------
            dict: chi, chi2, dof, p_value, expected (expected counts per bin at best fit)
            """
            counts = np.asarray(counts, dtype=float)
            bin_edges = np.asarray(bin_edges, dtype=float)
            total = counts.sum()
        
            bin_mids = 0.5 * (bin_edges[:-1] + bin_edges[1:])
            int_counts = np.round(counts).astype(int)
            samples = np.repeat(bin_mids, int_counts)
        
            # NOTE: this is scipy.stats.fit (the standalone function), not
            # argus.fit (the distribution method) -- only the former accepts
            # per-parameter `bounds`, which is what keeps chi from collapsing to
            # a degenerate near-zero value on awkward data.
            fit_result = _scipy_stats_fit(
                argus, samples,
                bounds={'chi': chi_bounds, 'loc': (0, 0), 'scale': (1, 1)},
            )
            chi_fit = float(fit_result.params.chi)
        
            expected = _expected_counts(chi_fit, bin_edges, total)
            chi2_val = _binned_chi2(counts, expected)
            dof = len(counts) - 1
            p_value = float(chi2_dist.sf(chi2_val, dof))
        
            return {
                'chi': chi_fit,
                'chi2': chi2_val,
                'dof': dof,
                'p_value': p_value,
                'expected': expected,
            }
        
        
        def rescale_bin_edges(bin_edges, low=None, high=None):
            """
            Rescale raw bin edges into [0, 1] so they're usable with fit_argus_chi2 /
            fit_argus_mle (which fix loc=0, scale=1).
        
            Parameters
            ----------
            bin_edges : array_like
                Your histogram's bin edges, in whatever physical units they're in
                (e.g. GeV, energy, mass, whatever your x-axis actually is).
            low, high : float, optional
                The physical domain to map to [0, 1]. Defaults to bin_edges[0] and
                bin_edges[-1] -- i.e. "my histogram already spans the full ARGUS
                support". Pass these explicitly if your histogram's range is
                narrower than the true physical cutoff (e.g. you binned data from
                5.20 to 5.29 GeV but the real kinematic endpoint is 5.30 GeV).
        
            Returns
            -------
            (scaled_edges, low, high) : the [0, 1]-scaled edges, plus the low/high
                values used (so you can keep them around -- you don't need to
                invert chi itself, since chi is a dimensionless shape parameter,
                but you WILL need low/high again if you want to plot the fitted
                PDF back in physical units).
            """
            bin_edges = np.asarray(bin_edges, dtype=float)
            low = bin_edges[0] if low is None else low
            high = bin_edges[-1] if high is None else high
            scaled_edges = (bin_edges - low) / (high - low)
            return scaled_edges, low, high
        
        
        def fit_argus(counts, bin_edges, low=None, high=None, method='chi2', **kwargs):
            """
            Convenience wrapper: rescales raw bin_edges to [0, 1], then runs one of
            the two fit strategies above. THIS is where the x-scaling actually
            happens -- fit_argus_chi2/fit_argus_mle themselves assume [0, 1] already.
        
            Parameters
            ----------
            counts : array_like
                Raw histogram counts, unchanged by rescaling.
            bin_edges : array_like
                Raw bin edges, in your data's physical units.
            low, high : float, optional
                Physical domain mapped to [0, 1]. See rescale_bin_edges.
            method : {'chi2', 'mle'}
                Which fit strategy to use.
            **kwargs :
                Passed through to the underlying fit function (e.g. chi_bounds for
                method='chi2').
        
            Returns
            -------
            dict, same keys as fit_argus_chi2/fit_argus_mle, plus 'low' and 'high'
            so you can map the fitted PDF back to physical units later:
                x_physical = low + x_scaled * (high - low)
                pdf_physical(x_physical) = argus.pdf(x_scaled, chi) / (high - low)
            """
            scaled_edges, low, high = rescale_bin_edges(bin_edges, low, high)
        
            if method == 'chi2':
                result = fit_argus_chi2(counts, scaled_edges, **kwargs)
            elif method == 'mle':
                result = fit_argus_mle(counts, scaled_edges, **kwargs)
            else:
                raise ValueError("method must be 'chi2' or 'mle'")
        
            result['low'] = low
            result['high'] = high
            return result
        
        
        def scan_low(counts, bin_edges, low_values, high=None, method='chi2', **kwargs):
            """
            Refit at each candidate `low` value and collect the results, so you can
            see how sensitive the fit is to your choice of `low` (and find the value
            that actually minimizes chi2).
        
            Parameters
            ----------
            counts, bin_edges : array_like
                Same as fit_argus.
            low_values : array_like
                Candidate values for `low` to try, e.g. np.linspace(5.15, 5.25, 21).
            high : float, optional
                Held fixed across the scan (defaults to bin_edges[-1], same as fit_argus).
            method : {'chi2', 'mle'}
            **kwargs :
                Passed through to fit_argus (e.g. chi_bounds).
        
            Returns
            -------
            list of dict: each is a fit_argus() result dict with an added
            'low_tested' key (== the 'low' actually used, kept for clarity even
            though it duplicates result['low']).
            """
            results = []
            for low_val in low_values:
                r = fit_argus(counts, bin_edges, low=low_val, high=high, method=method, **kwargs)
                r['low_tested'] = low_val
                results.append(r)
            return results
        
        
        def best_low(results):
            """Pick the result with the lowest chi2 from a scan_low() list."""
            return min(results, key=lambda r: r['chi2'])
        
        
        def plot_low_scan(results, ax=None):
            """
            Plot chi2 vs. the scanned `low` values, marking the minimum.
        
            Parameters
            ----------
            results : list of dict
                Output of scan_low().
            ax : matplotlib Axes, optional
        
            Returns
            -------
            ax
            """
            lows = np.array([r['low_tested'] for r in results])
            chi2s = np.array([r['chi2'] for r in results])
            best = best_low(results)
        
            if ax is None:
                _fig, ax = plt.subplots(figsize=(7, 4))
        
            ax.plot(lows, chi2s, 'o-', color='steelblue')
            ax.axvline(best['low_tested'], color='crimson', linestyle='--',
                    label=f"best low = {best['low_tested']:.4g}  (chi2={best['chi2']:.1f})")
            ax.set_xlabel('low')
            ax.set_ylabel('chi2')
            ax.legend()
            return ax
        
        
        def plot_argus_fit(counts, bin_edges, fit_result, ax=None, n_points=500,
                            data_label='Data', fit_label='ARGUS fit', show_stats=True):
            """
            Overlay the fitted ARGUS PDF on top of your histogram.
        
            Parameters
            ----------
            counts : array_like
                Your raw histogram counts (same ones passed to fit_argus/fit_argus_chi2/fit_argus_mle).
            bin_edges : array_like
                Your raw bin edges, in physical units (same ones you fitted with).
            fit_result : dict
                The dict returned by fit_argus(...) -- needs 'chi', and 'low'/'high'
                if you used fit_argus() (physical units). If you called
                fit_argus_chi2/fit_argus_mle directly on already-[0,1] data, just
                add fit_result['low'] = 0.0 and fit_result['high'] = 1.0 yourself
                before passing it in here.
            ax : matplotlib Axes, optional
                Axes to draw on; creates a new figure if omitted.
            n_points : int
                Number of points used to draw the smooth PDF curve.
            data_label, fit_label : str
                Legend labels.
            show_stats : bool
                If True, annotates the plot with chi, chi2/dof, and p-value.
        
            Returns
            -------
            ax : the matplotlib Axes drawn on.
            """
            counts = np.asarray(counts, dtype=float)
            bin_edges = np.asarray(bin_edges, dtype=float)
            total = counts.sum()
            chi_fit = fit_result['chi']
            low = fit_result.get('low', 0.0)
            high = fit_result.get('high', 1.0)
        
            if ax is None:
                _fig, ax = plt.subplots(figsize=(7, 5))
        
            # --- histogram (as counts, i.e. bar heights, not density) ---
            bin_widths = np.diff(bin_edges)
            bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
            ax.bar(bin_centers, counts, width=bin_widths, alpha=0.5,
                edgecolor='black', linewidth=0.5, label=data_label)
        
            # --- smooth ARGUS curve, scaled from density to expected counts/bin ---
            # pdf_physical(x) = argus.pdf(x_scaled, chi) / (high - low)   [Jacobian]
            # curve in "counts" units = total * mean_bin_width * pdf_physical(x)
            x_phys = np.linspace(low, high, n_points)
            x_scaled = (x_phys - low) / (high - low)
            pdf_scaled = argus.pdf(x_scaled, chi_fit)
            mean_bin_width = np.mean(bin_widths)
            y_counts = total * mean_bin_width * pdf_scaled / (high - low)
        
            ax.plot(x_phys, y_counts, color='crimson', linewidth=2, label=fit_label)
        
            ax.set_xlabel('x')
            ax.set_ylabel('Counts')
            ax.legend()
        
            if show_stats:
                chi2_val = fit_result.get('chi2')
                dof = fit_result.get('dof')
                p_value = fit_result.get('p_value')
                text = rf"$\chi_{{\rm ARGUS}}$ = {chi_fit:.3f}"
                if chi2_val is not None and dof is not None:
                    text += f"\n$\\chi^2$/dof = {chi2_val:.1f}/{dof}"
                if p_value is not None:
                    text += f"\np = {p_value:.3f}"
                ax.text(0.03, 0.97, text, transform=ax.transAxes,
                        va='top', ha='left', fontsize=10,
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
            return ax

        r1 = fit_argus(n, xe, low=None, high=5.3, method = 'chi2', chi_bounds=(0.05, 20.0))
        print(f"[chi2-minimization] chi = {r1['chi']:.4f}  "
            f"chi2 = {r1['chi2']:.2f}  dof = {r1['dof']}  p = {r1['p_value']:.4f}")
    
        r2 = fit_argus(n, xe, low=None, high=5.3, method = 'mle', chi_bounds=(0.001, 50.0))
        print(f"[MLE + chi2 check ] chi = {r2['chi']:.4f}  "
            f"chi2 = {r2['chi2']:.2f}  dof = {r2['dof']}  p = {r2['p_value']:.4f}")
        
        result = r1
        ax = plot_argus_fit(n, xe, result)
        plt.show()
        result = r2
        ax = plot_argus_fit(n, xe, result)
        plt.show()

    if False:
        x_plot = np.linspace(bins[0], bins[-1], 200)

        # ----------------------------------------------------
        # 1. Constant Fit (Degree 0)
        # ----------------------------------------------------
        c_p, c_err, c_chi, c_ndf = fit_and_analyze(bin_centers, counts[1], counts998, counts1005, degree=0)
        print(f"--- Constant Fit ---")
        print(f"Parameter c = {c_p[0]:.3f} ± {c_err[0]:.3f}")
        print(f"Chi-Sq = {c_chi:.2f}, NDF = {c_ndf}, Chi-Sq/NDF = {c_chi/c_ndf:.4f}\n")
        plt.plot(x_plot, constant_model(x_plot, *c_p), label='Constant Fit',linewidth=3, linestyle=':')

        # ----------------------------------------------------
        # 2. Linear Fit (Degree 1)
        # ----------------------------------------------------
        l_p, l_err, l_chi, l_ndf = fit_and_analyze(bin_centers, counts[1], counts998, counts1005, degree=1)
        print(f"--- Linear Fit ---")
        print(f"Parameter m = {l_p[0]:.3f} ± {l_err[0]:.3f}")
        print(f"Parameter c = {l_p[1]:.3f} ± {l_err[1]:.3f}")
        print(f"Chi-Sq = {l_chi:.2f}, NDF = {l_ndf}, Chi-Sq/NDF = {l_chi/l_ndf:.4f}\n")
        plt.plot(x_plot, linear_model(x_plot, *l_p), label='Linear Fit', linestyle='--')

        # ----------------------------------------------------
        # 3. Quadratic Fit (Degree 2)
        # ----------------------------------------------------
        q_p, q_err, q_chi, q_ndf = fit_and_analyze(bin_centers, counts[1], counts998, counts1005, degree=2)
        print(f"--- Quadratic Fit ---")
        print(f"Parameter a = {q_p[0]:.4f} ± {q_err[0]:.4f}")
        print(f"Parameter b = {q_p[1]:.3f} ± {q_err[1]:.4f}")
        print(f"Parameter c = {q_p[2]:.3f} ± {q_err[2]:.4f}")
        print(f"Chi-Sq = {q_chi:.2f}, NDF = {q_ndf}, Chi-Sq/NDF = {q_chi/q_ndf:.4f}\n")
        plt.plot(x_plot, quadratic_model(x_plot, *q_p), label='Quadratic Fit', linewidth=2)

        plt.legend()
        plt.xlabel("m_ES")
        plt.ylabel("Weighted Counts")
        plt.title(f'Weighted 998 & 1005, m_ES, |DeltaE|<0.07, {pidChoiceDict[pidChoice]}, no bcts / bctt')#, BCosThetaT < 0.82, BCosThetaS < 0.95')
        plt.show()

    data = blindD
    bCanMask = (ak.num(data['BMass'])==1)
    d1b = data[bCanMask]
    indices_and_maps = get_info_for_PID_masks(d1b, verbosity=0)
    mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2 = PID_masks(indices_and_maps, \
                lam1p_selector=myPMask, \
                lam1pi_selector=myPiMask, \
                lam2p_selector=myPMask, \
                lam2pi_selector=myPiMask, \
                verbosity=0)
    mes = d1b['BpostFitMes']
    de = d1b['BpostFitDeltaE']
    meslo = 5.2
    mesloS = 5.27
    meshiS = 5.3
    DeltaElo = -0.2
    DeltaEhi = 0.2
    DeltaEloS = -0.07
    DeltaEhiS = 0.07
    mask_pid = mask_bool_pionL1 & mask_bool_protonL1 & mask_bool_pionL2 & mask_bool_protonL2
    preSel_fit = (mes>meslo)&(de<DeltaEhi)&(de>DeltaElo)
    lamconflsig = d1b['Lambda0postFitFlightSignificance']
    lamuncmass = d1b['Lambda0_unc_Mass']
    BCTT = d1b['BCosThetaT']
    BCTS = d1b['BCosThetaS']
    #TMA = d1b['thrustMagAll']
    #BP2 = d1b['BLegendreP2']
    lammass_world_average = 1.115683
    width = 0.003
    lo = lammass_world_average - width
    hi = lammass_world_average + width
    mask_fl = (lamconflsig>=minSigCut)
    mask_upper_fl = (lamconflsig>=maxSigCut)
    flightSigMask = (ak.all(mask_fl,axis=1))&(ak.any(mask_upper_fl, axis=1))
    lamuncmassMask = (lamuncmass[:,0]>=lo)&(lamuncmass[:,0]<=hi)& (lamuncmass[:,1]>=lo)&(lamuncmass[:,1]<=hi)
    vetoMask = apply_antibaryon_veto(d1b, pps, 'LooseKMProtonSelection')
    bcts_cut = 95
    bctt_cut = 82
    bcttMask = (BCTT<bctt_cut/100) 
    bctsMask = (BCTS<bcts_cut/100)
    deMask = (de >= DeltaEloS) & (de <= DeltaEhiS)
    cut_mask = preSel_fit&deMask&mask_pid&lamuncmassMask&flightSigMask&vetoMask#&bcttMask&bctsMask
    mesP = mes[cut_mask]

    bkgList = ak.to_numpy(ak.flatten(mesP))
    labels = ['Blinded Data']
    colors = ['blue']

    bin_edges = np.linspace(5.2, 5.27, 28)


    plt.figure(figsize = (8,6))
    #n, xe
    counts, bins, patches = plt.hist(
            bkgList, 
            bins=bin_edges,
            label=labels,
            color=colors, 
            edgecolor='black',
            alpha = 0.6
        )
    
    bin_centers = (bins[:-1] + bins[1:]) / 2  

    if True:
        n = counts
        xe = bins

        #counts, _ = np.histogram(bkgList, bins=bin_edges)


        def linear_cdf(x, slope):
            """ 
            A vectorized mathematical model. 
            'slope' is the parameter iminuit will change.
            """
            a, b = 5.20, 5.27
            u = (x - a) / (b - a)  # Normalized x between 0 and 1
            
            # Mathematical formula for a varying linear background CDF
            return u + slope * (u**2 - u)
        def quadratic_cdf(x, alpha, beta):
            """
            Vectorized Quadratic CDF. 
            alpha controls the linear slope tilt.
            beta controls the quadratic curvature (parabola).
            """
            # Map x onto a [0, 1] domain
            A, B = 5.20, 5.27
            u = (x - A) / (B - A)
            
            # Vectorized analytical integral equation
            return u + alpha * (u**2 - u) + beta * (2 * u**3 - 3 * u**2 + u)
        def normExpon_cdf(xe, z, mu, sigma, tau):
            return z * truncnorm.cdf(xe, *xr, mu, sigma) + (1 - z) * truncexpon.cdf(xe, *xr, 0, tau)
        def argus_cdf_windowedQ(xval, chi):
            m0 = 5.29
            xmin = 5.2
            xmax = 5.27
            raw = argus.cdf(xval, chi, loc=0, scale=m0)
            lo  = argus.cdf(xmin, chi, loc=0, scale=m0)
            hi  = argus.cdf(xmax, chi, loc=0, scale=m0)
            return (raw - lo) / (hi - lo)
        def argus_cdf_windowed_extendedQ(xval, s, chi):
            m0 = 5.29
            xmin = 5.2
            xmax = 5.27
            raw = argus.cdf(xval, chi, loc=0, scale=m0)
            lo  = argus.cdf(xmin, chi, loc=0, scale=m0)
            hi  = argus.cdf(xmax, chi, loc=0, scale=m0)
            return s * (raw - lo) / (hi - lo)



        cx = bin_centers
        vn = np.where(n != 0, n, 1)

        #n = n[:-12]
        #vn = vn[:-12]
        #xe = xe[:-12]

        

        if plotVal == 0:
            c = cost.BinnedNLL(np.transpose((n,vn)), xe, argus_cdf_windowed)
            m = Minuit(c, chi = 1.2)
            m.strategy = 2
            m.limits['chi'] = (0, 20)
            m.migrad()
            m.hesse()
            model_cdf = argus_cdf_windowed(xe, m.values['chi'])
            expected = n.sum()*np.diff(model_cdf) 
            plt.errorbar(cx, n, np.sqrt(vn), fmt="ok")                            # expected counts per bin
            plt.stairs(expected, xe, fill=False, color="red", linewidth=2.5)
            plt.show()
            #m.visualize()
            print('Chi:')
            print(m.values['chi'])
            m.minos()
            lo = m.merrors["chi"].lower   # negative number — this is an offset, not the bound itself
            hi = m.merrors["chi"].upper   # positive number
            print(f"chi = {m.values['chi']:.4f} [{m.values['chi']+lo:.4f}, {m.values['chi']+hi:.4f}]")
            chi_hat = m.values['chi']
            sigma_chi = 10
            m0, xmin, xmax = 5.29, 5.2, 5.27
            N_total = float(n.sum())
            a, b = 5.2, 5.27
            pred, pred_err = argus_events_in_region_with_error(chi_hat, sigma_chi, m0, xmin, xmax, a, b, N_total)
            print(f"predicted ARGUS events in [{a}, {b}]: {pred:.1f} +/- {pred_err:.1f}")
            print(f"actual count in that region:    {sum(n)}")
            a, b = 5.27, 5.29
            pred, pred_err = argus_events_in_region_with_error(chi_hat, sigma_chi, m0, xmin, xmax, a, b, N_total)
            print(f"predicted ARGUS events in [{a}, {b}]: {pred:.1f} +/- {pred_err:.1f}")

        if plotVal == 1:
            c = cost.ExtendedBinnedNLL(np.transpose((n,vn)), xe, argus_cdf_windowed_extended)
            m = Minuit(c, s=20, chi = 5)
            m.strategy = 2
            m.limits['chi'] = (0, 15)
            m.limits['s'] = (1, 27)

            m.migrad()
            m.hesse()
            model_cdf = argus_cdf_windowed_extended(xe, m.values['s'], m.values['chi'])
            expected = np.diff(model_cdf) 
            plt.errorbar(cx, n, np.sqrt(vn), fmt="ok")                            # expected counts per bin
            plt.stairs(expected, xe, fill=False, color="red", linewidth=2.5)
            plt.show()
            #m.visualize()
            print('Chi:')
            print(m.values['chi'])
            print(m.values['s'])
            m.minos()
            lo = m.merrors["chi"].lower   # negative number — this is an offset, not the bound itself
            hi = m.merrors["chi"].upper   # positive number
            print(f"chi = {m.values['chi']:.4f} [{m.values['chi']+lo:.4f}, {m.values['chi']+hi:.4f}]")
            chi_hat_ext = m.values['chi']
            s = m.values['s']
            a, b = 5.2, 5.27
            predExt = extended_argus_events_in_region(chi_hat_ext, a, b, s)
            print(f"predicted extended ARGUS events in [{a}, {b}]: {predExt:.1f}")
            print(f"actual count in that region:    {sum(n)}")
            a, b = 5.27, 5.29
            predExt = extended_argus_events_in_region(chi_hat_ext, a, b, s)
            print(f"predicted extended ARGUS events in [{a}, {b}]: {predExt:.1f}")

        if plotVal == 2:
            c = cost.BinnedNLL(np.transpose((n,vn)), xe, normExpon_cdf)
            m = Minuit(c, z=0.4, mu = 5.1, sigma = 0.11, tau = 2)
            m.limits["z"] = (0, 1)
            m.limits["mu"] = (0,None)
            m.limits["sigma", "tau"] = (0.01, None)
            m.migrad()
            m.hesse()
            model_cdf = normExpon_cdf(xe, m.values['z'], m.values['mu'],m.values['sigma'], m.values['tau'])
            expected = n.sum()*np.diff(model_cdf) 
            plt.errorbar(cx, n, np.sqrt(vn), fmt="ok")                            # expected counts per bin
            plt.stairs(expected, xe, fill=False, color="red", linewidth=2.5)
            plt.show()
            #m.visualize()
            print('mu, sigma, tau:')
            print(m.values['mu'])
            print(m.values['sigma'])
            print(m.values['tau'])
            z = m.values['z']
            low = 5.2
            high = 5.27
            mu= m.values['mu']
            sigma = m.values['sigma']
            tau = m.values['tau']
            N_total = np.sum(n)
            integration_start = 5.2
            integration_end = 5.27
            val, err = integrate_mixture_cdf(z, low, high, mu, sigma, tau, N_total, integration_start, integration_end)
            print(f"NORM + EXPON MC\nIntegral value in [{integration_start}, {integration_end}]: {val}")
            integration_start = 5.27
            integration_end = 5.29
            val, err = integrate_mixture_cdf(z, low, high, mu, sigma, tau, N_total, integration_start, integration_end)
            print(f"NORM + EXPON MC\nIntegral value in [{integration_start}, {integration_end}]: {val}")

        if plotVal == 3:
            c = cost.BinnedNLL(np.transpose((n,vn)), xe, linear_cdf)
            m = Minuit(c, slope = 0)
            m.limits["slope"] = (None, 0) 
            m.migrad()
            m.hesse()
            model_cdf = linear_cdf(xe, m.values['slope'])
            expected = n.sum()*np.diff(model_cdf) 
            plt.errorbar(cx, n, np.sqrt(vn), fmt="ok")                            # expected counts per bin
            plt.stairs(expected, xe, fill=False, color="red", linewidth=2.5)
            plt.show()
            #m.visualize()
            print('Slope:')
            print(m.values['slope'])
        if plotVal == 4:
            c = cost.BinnedNLL(np.transpose((n,vn)), xe, quadratic_cdf)
            m = Minuit(c, alpha = 0, beta = 0)
            m.limits["alpha"] = (-1, 1.0)
            m.limits["beta"] = (-1, 1)
            m.migrad()
            m.hesse()
            model_cdf = quadratic_cdf(xe, m.values['alpha'],m.values['beta'])
            expected = n.sum()*np.diff(model_cdf) 
            plt.errorbar(cx, n, np.sqrt(vn), fmt="ok")                            # expected counts per bin
            plt.stairs(expected, xe, fill=False, color="red", linewidth=2.5)
            plt.show()
            #m.visualize()
            print('Alpha, beta:')
            print(m.values['alpha'])
            print(m.values['beta'])

        m.migrad()
        m.hesse()

        y_errs = np.sqrt(n)

        plt.show()
        m
        print(m)